# Phase 02 — Conclusions and Report Tables

Per the KT2 guide (Faza A — Zaključci): one **summary table** of the best model per task (commercial vs. award), **with and without** hyperparameter optimization, plus a short paragraph answering: *which feature set and model give the best F1/AUC and why?* Also compare **ensemble (stacking)** vs. best single model for the report.

## Load results

Run **`04_model_improvements.ipynb`** first (it saves `data/processed/phase2_results.pkl`). This notebook loads that file so it runs **standalone**.

In [1]:
import pickle
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path('../data') / 'processed'
pkl_path = PROCESSED_DIR / 'phase2_results.pkl'
if not pkl_path.exists():
    raise FileNotFoundError(f'Run 04_model_improvements first. Missing: {pkl_path}')

with open(pkl_path, 'rb') as f:
    _bundle = pickle.load(f)

all_results = _bundle.get('all_results', {})
tuning_results = _bundle.get('tuning_results', {})
ensemble_results = _bundle.get('ensemble_results', {})

TUNED_DISPLAY = {'LogReg': 'LogReg (tuned)', 'RF': 'RF (tuned)', 'XGB': 'XGB (tuned)',
                'SVM_linear': 'SVM linear (tuned)', 'SVM_rbf': 'SVM RBF (tuned)'}

## Build: best model per task (baseline vs tuned)

all_results (from 04) already includes LR, RF, XGB, and SVM. We pick the best baseline and best tuned model per task/feature set. Table rows: Task, Feature set, Best model (baseline), F1 (baseline), AUC (baseline), Best model (tuned), F1 (tuned), AUC (tuned).

In [2]:
def best_in_results(results_dict, prefer_f1_opt=True):
    """From a dict model_name -> {test_f1, test_f1_opt, test_auc, ...} return best model name and metrics."""
    if not results_dict:
        return None, None, None, None
    f1_key = 'test_f1_opt' if prefer_f1_opt else 'test_f1'
    best_name = None
    best_f1 = best_auc = -1
    for name, r in results_dict.items():
        if not isinstance(r, dict) or 'test_f1' not in r:
            continue
        f1 = r.get(f1_key) or r.get('test_f1') or 0
        auc = r.get('test_auc') or 0
        if f1 > best_f1:
            best_f1, best_auc, best_name = f1, auc, name
    return best_name, best_f1, best_auc, results_dict.get(best_name, {})

rows = []
for target, label in [('is_commercial', 'Commercial'), ('is_award_winner', 'Award')]:
    for set_name in ['7_+runtime', '8_+vote_revenue']:
        base = all_results.get(set_name, {}).get(target, {})
        tuned = tuning_results.get(set_name, {}).get(target, {})
        b_name, b_f1, b_auc, _ = best_in_results(base)
        t_name, t_f1, t_auc, _ = best_in_results(tuned)
        rows.append({
            'Task': label,
            'Feature set': set_name,
            'Best (baseline)': b_name or '—',
            'F1 (baseline)': f'{b_f1:.3f}' if b_f1 >= 0 else '—',
            'AUC (baseline)': f'{b_auc:.3f}' if b_auc >= 0 else '—',
            'Best (tuned)': (TUNED_DISPLAY.get(t_name, t_name) if t_name else '—'),
            'F1 (tuned)': f'{t_f1:.3f}' if t_f1 >= 0 else '—',
            'AUC (tuned)': f'{t_auc:.3f}' if t_auc >= 0 else '—',
        })

table_best = pd.DataFrame(rows)
print('Best model per task and feature set (baseline vs tuned)')
display(table_best)

Best model per task and feature set (baseline vs tuned)


,Task,Feature set,Best (baseline),F1 (baseline),AUC (baseline),Best (tuned),F1 (tuned),AUC (tuned)
0,Commercial,7_+runtime,SVM (linear),0.657,0.632,RF (tuned),0.664,0.631
1,Commercial,8_+vote_revenue,SVM (linear),0.657,0.632,RF (tuned),0.664,0.631
2,Award,7_+runtime,Random Forest,0.464,0.782,XGB (tuned),0.490,0.801
3,Award,8_+vote_revenue,XGBoost,0.527,0.826,LogReg (tuned),0.549,0.857


## Ensemble vs best single model

For 7_+runtime (and optionally 8_+vote_revenue), compare Stacking (LR) to the best single model.

In [3]:
ensemble_comparison = []
for set_name in ['7_+runtime', '8_+vote_revenue']:
    if set_name not in ensemble_results:
        continue
    base = all_results.get(set_name, {})
    for target, label in [('is_commercial', 'Commercial'), ('is_award_winner', 'Award')]:
        base_t = base.get(target, {})
        b_name, b_f1, b_auc, _ = best_in_results(base_t)
        ens = ensemble_results[set_name].get(target, {})
        stack = ens.get('Stacking (LR)', {})
        if not stack:
            continue
        ensemble_comparison.append({
            'Feature set': set_name,
            'Task': label,
            'Best single': b_name or '—',
            'Best single F1': f'{b_f1:.3f}' if b_f1 >= 0 else '—',
            'Stacking F1': f"{stack.get('test_f1', 0):.3f}",
            'Stacking AUC': f"{stack.get('test_auc') or 0:.3f}",
        })

if ensemble_comparison:
    display(pd.DataFrame(ensemble_comparison))
else:
    print('Run notebook 04_model_improvements to populate ensemble_results.')

,Feature set,Task,Best single,Best single F1,Stacking F1,Stacking AUC
0,7_+runtime,Commercial,SVM (linear),0.657,0.578,0.645
1,7_+runtime,Award,Random Forest,0.464,0.349,0.766
2,8_+vote_revenue,Commercial,SVM (linear),0.657,0.578,0.645
3,8_+vote_revenue,Award,XGBoost,0.527,0.422,0.812


## Report paragraph: conclusions

Use the tables above to write **one short paragraph** for the final report (Metod / Rezultati / Zaključci), covering:

1. **Which feature set and model give the best F1/AUC** for commercial success and for award winner, and **why** (e.g. budget and runtime are strong for commercial; vote_count and revenue help award; SVM or XGB benefits from high-dimensional text).
2. **Gain from optimization:** tuning (RandomizedSearchCV) typically yields a small gain (e.g. +2–4% F1/AUC); report the numbers from the table.
3. **Ensembles:** Does stacking beat the best single model? Often the gain is small when base models are correlated; state the finding for your runs.

Example (replace with your numbers):

> *For commercial success, the best baseline was [XGB/LR/RF/SVM] on 7_+runtime (F1 ≈ 0.65, AUC ≈ 0.66). After hyperparameter tuning, [model] on [set] reached F1 ≈ 0.68 and AUC ≈ 0.70. For award winner, 8_+vote_revenue with [model] gave F1 ≈ 0.52 and AUC ≈ 0.80; tuning improved F1 to ~0.55. Stacking (LR) was [on par / slightly better / worse] than the best single model, which is consistent with [similar base classifiers / limited diversity].*

## Export table for the report

Save the main conclusions table as CSV for inclusion in the IEEE-format report.

### KT2 targets (automated check)

| Task | F1(opt) target | AUC target |
|------|----------------|------------|
| Commercial | ≥ 0.68 | ≥ 0.70 |
| Award | ≥ 0.55 | ≥ 0.82 |

In [4]:
def max_tuned_f1_auc(tuned_dict):
    best_f1, best_auc = -1.0, -1.0
    for r in tuned_dict.values():
        if not isinstance(r, dict) or 'test_f1' not in r:
            continue
        f1 = r.get('test_f1_opt') or r.get('test_f1') or 0
        auc = r.get('test_auc') or 0
        best_f1 = max(best_f1, f1)
        best_auc = max(best_auc, auc)
    return best_f1, best_auc

for feat, label in [('7_+runtime', 'Commercial vs Award @ 7_+runtime'), ('8_+vote_revenue', ' @ 8_+vote_revenue')]:
    if feat not in tuning_results:
        continue
    cf1, cauc = max_tuned_f1_auc(tuning_results[feat]['is_commercial'])
    af1, aauc = max_tuned_f1_auc(tuning_results[feat]['is_award_winner'])
    print(f'\n{label}')
    print(f'  Commercial: best tuned F1(opt)={cf1:.3f} (target ≥0.68)  AUC={cauc:.3f} (target ≥0.70)')
    print(f'    -> F1 {"PASS" if cf1 >= 0.68 else "BELOW"}, AUC {"PASS" if cauc >= 0.70 else "BELOW"}')
    print(f'  Award:      best tuned F1(opt)={af1:.3f} (target ≥0.55)  AUC={aauc:.3f} (target ≥0.82)')
    print(f'    -> F1 {"PASS" if af1 >= 0.55 else "BELOW"}, AUC {"PASS" if aauc >= 0.82 else "BELOW"}')


Commercial vs Award @ 7_+runtime
  Commercial: best tuned F1(opt)=0.664 (target ≥0.68)  AUC=0.631 (target ≥0.70)
    -> F1 BELOW, AUC BELOW
  Award:      best tuned F1(opt)=0.490 (target ≥0.55)  AUC=0.802 (target ≥0.82)
    -> F1 BELOW, AUC BELOW

 @ 8_+vote_revenue
  Commercial: best tuned F1(opt)=0.664 (target ≥0.68)  AUC=0.631 (target ≥0.70)
    -> F1 BELOW, AUC BELOW
  Award:      best tuned F1(opt)=0.549 (target ≥0.55)  AUC=0.857 (target ≥0.82)
    -> F1 BELOW, AUC PASS


In [5]:
FIGURES_DIR = Path('figures')
FIGURES_DIR.mkdir(exist_ok=True)
out_path = FIGURES_DIR / 'phase02_conclusions_best_models.csv'
table_best.to_csv(out_path, index=False)
print(f'Saved to {out_path}')

Saved to figures\phase02_conclusions_best_models.csv
